# Extract DEX data from Dune

Pull swap-level prices, per-swap gas, mint/burn liquidity events, chain gas, and
hourly USD token prices for the pair, then save one folder of CSVs under `S.data_dir`.
All parameters come from `arblib.config.STUDY`; install deps with
`pip install -r requirements.txt`.

In [ ]:
import os

from dotenv import load_dotenv

from arblib import config, data_io, dune_api
from arblib.config import STUDY as S

load_dotenv()
headers = dune_api.make_headers(os.environ["DUNE_API_KEY"])
params = S.collection_params
params

## Swaps

One swap query per DEX, each left-joined with that DEX's per-swap gas on
block / pool / tx / event index, saved under `S.swaps_dir`.

In [ ]:
merge_keys = ["evt_block_number", "pool", "evt_tx_hash", "evt_index"]

df_pancake_swap = dune_api.run_dune_saved_query(config.SWAP_QUERY_IDS["pancake"], params, headers, "Pancake")
df_gas_pancake = dune_api.run_dune_saved_query(config.GAS_QUERY_IDS["pancake_gas_per_swap"], params, headers, "Gas pancake")
if not df_pancake_swap.empty and not df_gas_pancake.empty:
    df_pancake_swap = (df_pancake_swap.merge(df_gas_pancake, on=merge_keys, how="left")
                       .sort_values("evt_block_number").reset_index(drop=True))

df_uniswap_swap = dune_api.run_dune_saved_query(config.SWAP_QUERY_IDS["uniswap"], params, headers, "Uniswap")
df_gas_uniswap = dune_api.run_dune_saved_query(config.GAS_QUERY_IDS["uniswap_gas_per_swap"], params, headers, "Gas uniswap")
if not df_uniswap_swap.empty and not df_gas_uniswap.empty:
    df_uniswap_swap = (df_uniswap_swap.merge(df_gas_uniswap, on=merge_keys, how="left")
                       .sort_values("evt_block_number").reset_index(drop=True))

data_io.save_dataframes(
    {config.SWAP_FILES["df_uniswap"]: df_uniswap_swap, config.SWAP_FILES["df_pancake"]: df_pancake_swap},
    S.swaps_dir,
)

## Liquidity

Mint / burn events per pool, one query per DEX, saved under `S.liquidity_dir`.

In [ ]:
df_uniswap_liq = dune_api.run_dune_saved_query(config.LIQUIDITY_QUERY_IDS["uniswap"], params, headers, "Uniswap liquidity")
df_pancake_liq = dune_api.run_dune_saved_query(config.LIQUIDITY_QUERY_IDS["pancake"], params, headers, "Pancake liquidity")
if not df_uniswap_liq.empty:
    df_uniswap_liq = df_uniswap_liq.sort_values("evt_block_number").reset_index(drop=True)
if not df_pancake_liq.empty:
    df_pancake_liq = df_pancake_liq.sort_values("evt_block_number").reset_index(drop=True)

data_io.save_dataframes(
    {config.LIQUIDITY_FILES["df_uniswap"]: df_uniswap_liq, config.LIQUIDITY_FILES["df_pancake"]: df_pancake_liq},
    S.liquidity_dir,
)

## Chain gas

Per-block base fee + utilization for the whole chain, saved under `S.gas_dir`.

In [ ]:
df_gas_chain = dune_api.run_dune_saved_query(config.GAS_QUERY_IDS["chain_gas_price"], params, headers, "Gas price")
if not df_gas_chain.empty:
    df_gas_chain = df_gas_chain.sort_values("block_number").reset_index(drop=True)

data_io.save_dataframes({config.GAS_FILES["chain_gas_price"]: df_gas_chain}, S.gas_dir)

## USD token prices

Hourly USD price for each token of the pair, saved under `S.prices_dir`.

In [ ]:
USD_token_prices = dune_api.run_dune_saved_query(config.USD_PRICE_QUERY_ID["USD_price"], params, headers, "USD token prices")
if not USD_token_prices.empty:
    USD_token_prices = USD_token_prices.sort_values(["hour", "contract_address"]).reset_index(drop=True)

data_io.save_dataframes({S.prices_path.name: USD_token_prices}, S.prices_dir)